<!-- nav -->
[← 8.09 · Cpp Types](08_09_Cpp_Types.ipynb) · [Home](../../README.md) · [8.11 · POSIX BRE →](08_11_POSIX_BRE.ipynb)

# 8.10 · Cpp RegExp

C++'s library reads regular expressions: `std::regex r("ab|c");` compiles its string argument as a pattern of
[re.grammar]'s grammar, ECMAScript's (ECMA-262, its third edition), as [re.grammar] modifies it. To C++ the pattern is
a string, a literal like any other: no C++ subterm is a regular expression, and none is guessed to be one
([PRI-025: never guess the schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-025)). The
pattern's own grammar is a dialect of its own, `Cpp RegExp`, whose holder holds one. Its kinds are ECMAScript's, named as
regexpp, ECMAScript's reference parser, names them. And a `regex` names its grammar among its flags: five of
[re.synopt]'s flags select a grammar POSIX defines, whose pattern is a term of POSIX's own dialects,
[8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) and [8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb).

This page builds a pattern from two characters up, then ECMAScript's constructs, then a `regex` and its flags, one
whose flags name a POSIX grammar among them; then what validation says, and the editions it covers. It starts from
`dialects()`; `S` is its holder namespace. `char` writes a character, of its code point, as a spec the builders call,
and `alternative` and `pattern` the two kinds every pattern is made of, once 08.10.1 has written them in full:

In [1]:
from mbse.Expressions.Cpp import Expressions as Cpp, RegExp as CppRegExp
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from mbse.Expressions.POSIX import BRE, ERE
from toolkit import dialects

store = dialects()
S = store.Expressions
char = lambda c: lambda u: u.character(lambda r: r.value(ord(c)))
alternative = lambda *elements: lambda u: u.alternative(lambda r: r.elements(list(elements)))
pattern = lambda *alternatives: lambda u: u.pattern(lambda r: r.alternatives(list(alternatives)))
XR = CppRegExp.Expressions
DIALECT = XR.DIALECT


def refused(simplify, expression):
    """What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
    none (`NotEvaluated`, a `ValueError`)."""
    try:
        result = simplify(expression)
        return result.evaluate() if hasattr(result, "evaluate") else result
    except ValueError as error:
        return str(error)

<a id="T08.10.1"></a>
## 08.10.1 · A pattern, `ab|c`

```text
ab|c
```

A pattern is a disjunction of alternatives, each a sequence of terms (ECMA-262's *Disjunction*, *Alternative*, *Term*):
`ab|c` is a `pattern` whose `alternatives` are two `alternative`s, the first of the characters `a` and `b`, the second
of `c`. A `character` holds its code point, `value`, as regexpp does, and its spelling where the text gives it one of
its own (`\x41`). `S.Cpp.RegExp.Expression` holds it; `CppRegExp.Text.ToText` prints it as ECMAScript writes it:

In [2]:
with store.transaction():
    abc = S.Cpp.RegExp.Expression(lambda e: e.root(lambda u: u.pattern(lambda p: p.alternatives([
        lambda a: a.alternative(lambda r: r.elements([
            lambda c: c.character(lambda r: r.value(ord("a"))), lambda c: c.character(lambda r: r.value(ord("b")))])),
        lambda a: a.alternative(lambda r: r.elements([lambda c: c.character(lambda r: r.value(ord("c")))]))]))))
print(CppRegExp.Text.ToText(abc), "|", Terms.kind_of(Terms.term(abc))[0], "of", Terms.dialect_of(Terms.term(abc)))
for t in Terms.walk(abc):
    print(" ", Terms.kind_of(t)[0], *[f"{k}={v}" for k, v in Terms.form(t).attributes.items()])
print(DIALECT.name(), DIALECT.tag, "| its kinds:", ", ".join(DIALECT.kinds()))
assert CppRegExp.Text.ToText(abc) == "ab|c"

ab|c | pattern of Expressions.Cpp.RegExp.Term
  pattern
  alternative
  character value=97
  character value=98
  alternative
  character value=99
Cpp RegExp cpp_regexp | its kinds: regex, pattern, alternative, group, capturinggroup, quantifier, characterclass, characterclassrange, characterset, character, assertion, backreference, exclass, collatingelement, equivalence, error, missing


<a id="T08.10.2"></a>
## 08.10.2 · ECMAScript's constructs

```text
^(a|\.)b*c+\d{2,3}[^a-z]$        (?:a)(?=b)x*?\b
```

Each construct of ECMAScript's grammar is a kind: an `assertion`, `^` and `$` (its `assertionKind`, `start` or `end`;
`\b`, `word`; `(?=b)`, a `lookahead` holding alternatives); a `capturinggroup` and a non-capturing `group`; a
`quantifier`, its `min` and `max` and whether it is `greedy`, repeating one `element`; a `characterset`, `\d` or `.`
(its `setKind`); a `characterclass`, `[^a-z]`, holding characters and `characterclassrange`s. The printer escapes what
ECMAScript reads as syntax, so the character `.` prints `\.`:

In [3]:
repeat = lambda element, least, most=None: lambda u: u.quantifier(
    lambda r: r.min(least).element(element) if most is None else r.min(least).max(most).element(element))
with store.transaction():
    constructs = S.Cpp.RegExp.Expression(lambda e: e.root(pattern(alternative(
        lambda u: u.assertion(lambda r: r.assertionKind("start")),
        lambda u: u.capturinggroup(lambda r: r.alternatives([alternative(char("a")), alternative(char("."))])),
        repeat(char("b"), 0), repeat(char("c"), 1),
        repeat(lambda u: u.characterset(lambda r: r.setKind("digit")), 2, 3),
        lambda u: u.characterclass(lambda r: r.negate(True).elements([
            lambda u: u.characterclassrange(lambda r: r.min(char("a")).max(char("z")))])),
        lambda u: u.assertion(lambda r: r.assertionKind("end"))))))
    ecmascript = S.Cpp.RegExp.Expression(lambda e: e.root(pattern(alternative(
        lambda u: u.group(lambda r: r.alternatives([alternative(char("a"))])),
        lambda u: u.assertion(lambda r: r.assertionKind("lookahead").alternatives([alternative(char("b"))])),
        lambda u: u.quantifier(lambda r: r.min(0).greedy(False).element(char("x"))),
        lambda u: u.assertion(lambda r: r.assertionKind("word"))))))
for expression in (constructs, ecmascript):
    print(CppRegExp.Text.ToText(expression))
    print(" ", ", ".join(dict.fromkeys(Terms.kind_of(t)[0] for t in Terms.walk(expression))))
assert CppRegExp.Text.ToText(constructs) == "^(a|\\.)b*c+\\d{2,3}[^a-z]$"
assert CppRegExp.Text.ToText(ecmascript) == "(?:a)(?=b)x*?\\b"

^(a|\.)b*c+\d{2,3}[^a-z]$
  pattern, alternative, assertion, capturinggroup, character, quantifier, characterset, characterclass, characterclassrange
(?:a)(?=b)x*?\b
  pattern, alternative, group, character, assertion, quantifier


<a id="T08.10.3"></a>
## 08.10.3 · A `regex` and its flags

```text
std::regex r("ab|c", std::regex::icase);
```

A `std::regex` is built of its pattern and its `syntax_option_type` flags. The dialect's `regex` kind holds both: its
`flags` as written, and its `pattern`. The flags name at most one grammar; none names ECMAScript's, the default, whose
pattern is a `Cpp RegExp` term. The printer writes the pattern alone, since the flags are the constructor's other
argument. Which dialect each grammar's pattern is of is data the dialect holds, `GRAMMAR_DIALECTS`:

In [4]:
with store.transaction():
    insensitive = S.Cpp.RegExp.Expression(lambda e: e.root(lambda u: u.regex(lambda r: r.flags(["icase"])
        .pattern(lambda t: t.cpp_regexp(pattern(alternative(char("a"), char("b")), alternative(char("c"))))))))
form = Terms.form(Terms.term(insensitive))
print(CppRegExp.Text.ToText(insensitive), "| flags:", ", ".join(form.attributes["flags"].values),
      "| its grammar:", XR.grammar(form.attributes["flags"].values), "| its pattern:", Terms.dialect_of(form.arguments[0]))
for grammar, dialect in XR.GRAMMAR_DIALECTS.items():
    print(f"  {grammar:10}", dialect)
assert DIALECT.validate(insensitive) == [] and CppRegExp.Text.ToText(insensitive) == "ab|c"

ab|c | flags: icase | its grammar: ECMAScript | its pattern: Expressions.Cpp.RegExp.Term
  ECMAScript Cpp.RegExp
  basic      POSIX.BRE
  extended   POSIX.ERE
  awk        POSIX.ERE
  grep       POSIX.BRE
  egrep      POSIX.ERE


<a id="T08.10.4"></a>
## 08.10.4 · A POSIX grammar's pattern is POSIX's

```text
std::regex r("^(a)b+\n$", std::regex::extended);        std::regex r("^\\(a\\)b\\{1,\\}\n$", std::regex::basic);
```

[re.synopt]'s `basic` and `grep` select POSIX's basic regular expressions, and `extended`, `awk` and `egrep` its
extended ones, grammars [re.grammar] does not define, but POSIX does (IEEE Std 1003.1, XBD 9). So a `regex` whose flags
name one holds a pattern of POSIX's dialect, `POSIX BRE` or `POSIX ERE`, as a C++ term is read under the ABI its holder
names, and `Cpp RegExp` reads it as that dialect reads it. The same tree, a start anchor, a group of `a`, `b` repeated
at least once, a newline, an end anchor, is a POSIX ERE term under `extended`, printed as an ERE writes it, a POSIX BRE
term under `basic`, printed as a BRE writes it, with its backslashes, and an ERE under `awk`, which writes a newline as C
does, `\n`, where an ERE writes the newline itself (`⏎` below):

In [5]:
anchored = pattern(alternative(
    lambda u: u.assertion(lambda r: r.assertionKind("start")),
    lambda u: u.capturinggroup(lambda r: r.alternatives([alternative(char("a"))])),
    lambda u: u.quantifier(lambda r: r.min(1).element(char("b"))), char("\n"),
    lambda u: u.assertion(lambda r: r.assertionKind("end"))))
with store.transaction():
    posix = {flag: S.Cpp.RegExp.Expression(lambda e, flag=flag, branch=branch: e.root(lambda u: u.regex(
        lambda r: r.flags([flag]).pattern(lambda t: getattr(t, branch)(anchored))))) for flag, branch in [
        ("extended", "posix_ere"), ("basic", "posix_bre"), ("awk", "posix_ere")]}
for flag, regex in posix.items():
    held = Terms.form(Terms.term(regex)).arguments[0]
    print(f"{flag:8}", f"{CppRegExp.Text.ToText(regex).replace(chr(10), '⏎'):18}", "| its pattern:", DIALECT.owner(held).name(),
          "|", "; ".join(DIALECT.validate(regex)) or "valid")
assert [CppRegExp.Text.ToText(r) for r in posix.values()] == ["^(a)b+\n$", "^\\(a\\)b\\{1,\\}\n$", "^(a)b+\\n$"]

extended ^(a)b+⏎$           | its pattern: POSIX ERE | valid
basic    ^\(a\)b\{1,\}⏎$    | its pattern: POSIX BRE | valid
awk      ^(a)b+\n$          | its pattern: POSIX ERE | valid


What C++ selects of POSIX is checked under the regex. A grammar's pattern is a term of its dialect; and what POSIX.1-2024
added, which the editions C++ cites lack, is refused under C++'s grammars, though POSIX's dialects hold it: a BRE's
alternation, `\|`, and grep's list of patterns, which only `grep` and `egrep` read:

In [6]:
with store.transaction():
    wrong = {source: S.Cpp.RegExp.Expression(lambda e, flags=flags, spec=spec: e.root(lambda u: u.regex(
        lambda r: r.flags(flags).pattern(spec)))) for source, flags, spec in [
        ("ERE's term under basic", ["basic"], lambda t: t.posix_ere(pattern(alternative(char("a"))))),
        ("ab\\|c under basic", ["basic"], lambda t: t.posix_bre(pattern(alternative(char("a"), char("b")), alternative(char("c"))))),
        ("a list under awk", ["awk"], lambda t: t.posix_ere(lambda u: u.patternlist(lambda r: r.patterns([
            pattern(alternative(char("a"))), pattern(alternative(char("b")))])))),
        ("basic and extended", ["basic", "extended"], lambda t: t.posix_bre(pattern(alternative(char("a"))))),
    ]}
for source, regex in wrong.items():
    print(f"{source:20}", "|", "; ".join(DIALECT.validate(regex)))
assert DIALECT.validate(wrong["ERE's term under basic"]) == [
    "the basic grammar's pattern is a POSIX BRE term, got a POSIX ERE term"]

ERE's term under basic | the basic grammar's pattern is a POSIX BRE term, got a POSIX ERE term
ab\|c under basic    | alternation is not in the basic grammar
a list under awk     | a patternlist is not in the awk grammar
basic and extended   | a regex has at most one grammar element, got basic, extended


<a id="T08.10.5"></a>
## 08.10.5 · Checked, binding nothing, not simplified

Validation checks what each kind's terms may be, as ECMAScript's grammar has them: a quantifier repeats an atom, so a
quantifier of a quantifier, `a+*`, is refused, as is a quantifier's `max` below its `min`. A capturing group binds no
name a term scopes: groups are numbered, and a backreference, `\1`, names one by its number, so a pattern needs nothing
from its scope. No rule simplifies a regular expression: C++'s rules leave one in the residual, named by its dialect:

In [7]:
with store.transaction():
    checked = {
        "a+*": S.Cpp.RegExp.Expression(lambda e: e.root(pattern(alternative(repeat(repeat(char("a"), 1), 0))))),
        "a{3,2}": S.Cpp.RegExp.Expression(lambda e: e.root(pattern(alternative(repeat(char("a"), 3, 2))))),
        "(a)\\1": S.Cpp.RegExp.Expression(lambda e: e.root(pattern(alternative(
            lambda u: u.capturinggroup(lambda r: r.alternatives([alternative(char("a"))])),
            lambda u: u.backreference(lambda r: r.ref(1)))))),
    }
for source, expression in checked.items():
    print(f"{source:7}", "|", "; ".join(DIALECT.validate(expression)) or "valid",
          "| needs:", ", ".join(sorted(Symbolics.free(expression, DIALECT))) or "nothing")
print(refused(Cpp.Simplifiers.OfAny, insensitive))
assert CppRegExp.Text.ToText(checked["(a)\\1"]) == "(a)\\1"

a+*     | argument 0: argument 0: a quantifier's element cannot hold a quantifier | needs: nothing
a{3,2}  | argument 0: argument 0: a quantifier's max cannot be less than its min, got 2 and 3 | needs: nothing
(a)\1   | valid | needs: nothing
the root is not a literal: Cpp RegExp regex is not simplified


<a id="T08.10.6"></a>
## 08.10.6 · Its editions

`Cpp RegExp` covers C++'s three editions, each by its designation and its common name, every one of its kinds in each,
since [re.grammar] has not changed since C++11 ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)).
The editions of the POSIX grammars a regex selects are POSIX's own, which their pages read:

In [8]:
regexp = store.Instances.Expressions.Cpp.RegExp  # the dialect, by its global name
for designation, edition in Standards.standards(regexp).items():
    print(designation, "|", ", ".join(Standards.tags(edition)[1:]), "|", len(Standards.kinds(edition)), "kinds")
assert Standards.find(regexp, "C++17") is Standards.standards(regexp)["ISO/IEC 14882:2017"]

ISO/IEC 14882:2017 | C++17 | 15 kinds
ISO/IEC 14882:2020 | C++20 | 15 kinds
ISO/IEC 14882:2024 | C++23 | 15 kinds


## What you learned

- **A C++ regular expression is a `Cpp RegExp` term**, held by its own holder, since to C++ a pattern is a string.
- **Its kinds are ECMAScript's**, as regexpp names them: a pattern of alternatives, assertions, groups, quantifiers,
  character sets and classes, backreferences.
- **A `regex` keeps its flags as written**, and its flags name its grammar: ECMAScript's by default, or one of POSIX's,
  whose pattern is a term of [8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) or [8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb), printed
  as POSIX writes it.
- **What C++ selects of POSIX is checked under the regex**, and nothing is simplified.

Next, POSIX's basic regular expressions, [8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb).

<!-- nav -->
[← 8.09 · Cpp Types](08_09_Cpp_Types.ipynb) · [Home](../../README.md) · [8.11 · POSIX BRE →](08_11_POSIX_BRE.ipynb)